<h1> <strong> <center> NLP Project -- Module 3: Analysis and Inference of Financial Reports </center> </strong> </h1>

<h4> <strong>  1)  Library Imports </strong> </h4>

In [ ]:
import glob
import time
import os
import re
import getpass
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from unstructured.partition.html import partition_html
from langchain_core.tools import StructuredTool
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain_experimental.tools import PythonREPLTool
from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field
from sec_edgar_downloader import Downloader

from unstructured.partition.html import partition_html
from unstructured.chunking.title import chunk_by_title

from bs4 import BeautifulSoup

In [ ]:
# Tavily was prompted for in the original notebook but never used by any tool.
# Only the Google AI key is required.
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")


<h4> <strong>  2) SEC Filings Download Setup: </strong> </h4>

* defines project and data directories for storing SEC filings
* specifies a list of company tickers and the number of recent years to download
* uses a `Downloader` utility to fetch the last 5 years of 10-K filings for each ticker

In [ ]:
base_path = os.environ.get("FINRAG_DATA_ROOT", "./data")
project_folder = os.path.join(base_path, "Financial_Analyzer_Project")
data_folder = os.path.join(project_folder, "sec_filings")
os.makedirs(data_folder, exist_ok=True)

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "META", "NVDA", "NFLX", "JPM", "V"]
user_email = os.environ.get("SEC_CONTACT_EMAIL", "")
years_limit = 5

print(f"\n Downloading last {years_limit} years of 10-K filings")

dl = Downloader("MyFinancialApp", user_email, data_folder)

for ticker in tickers:
    try:
        print(f"  {ticker}")
        dl.get("10-K", ticker, limit=years_limit)
    except Exception as e:
        print(f"   {ticker}: {e}")

print("\n SEC Download Complete")

<h4> <strong>  3) SEC Filings Parsing, Chunking, and Semantic Indexing: </strong> </h4>

* defines a **Chroma vector database path** and embedding model for semantic representation of SEC filings
* implements utilities to extract metadata from file paths, convert HTML tables to Markdown, and parse 10-K sections from raw SEC filing text
* iterates over downloaded `.txt` filings, cleans and splits content into overlapping chunks, wraps each chunk with metadata, and indexes them into Chroma for retrieval-based applications


In [ ]:
db_path = os.path.join(project_folder, "chroma_db_financial_semantic")
embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

def get_metadata_from_path(file_path):
    parts = file_path.split(os.sep)
    try:
        if "sec-edgar-filings" in parts:
            root_idx = parts.index("sec-edgar-filings")
            ticker = parts[root_idx + 1] 
            accession_dir = parts[root_idx + 3]
            year_short = accession_dir.split('-')[1]
            year = int("20" + year_short)
            return {"ticker": ticker, "year": year}
    except Exception:
        pass
    return {"ticker": "UNKNOWN", "year": 0}

def html_to_markdown_table(soup):
    for table in soup.find_all('table'):
        md_table = "\n"
        for row in table.find_all('tr'):
            cells = row.find_all(['td', 'th'])
            clean_cells = [c.get_text(strip=True).replace("|", "-") for c in cells]
            if any(clean_cells): 
                md_table += "| " + " | ".join(clean_cells) + " |\n"
        table.replace_with(md_table + "\n")
    return soup

def parse_sec_structured(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        documents = content.split('<DOCUMENT>')
        for doc in documents:
            if '<TYPE>10-K' in doc[:2000].upper():
                start_idx = doc.find('<TEXT>')
                end_idx = doc.find('</TEXT>')
                if start_idx != -1 and end_idx != -1:
                    raw_html = doc[start_idx + 6 : end_idx]
                    soup = BeautifulSoup(raw_html, "html.parser")
                    soup = html_to_markdown_table(soup)
                    return soup.get_text(separator="\n")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

txt_files = glob.glob(os.path.join(data_folder, "**", "*.txt"), recursive=True)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)

for i, file_path in enumerate(txt_files):
    meta = get_metadata_from_path(file_path)
    print(f"[{i+1}/{len(txt_files)}] Processing {meta['ticker']} {meta['year']} ")
    
    clean_text = parse_sec_structured(file_path)
    if clean_text:
        chunks = text_splitter.split_text(clean_text)
        batch_docs = [Document(page_content=c, metadata=meta) for c in chunks]
        Chroma.from_documents(batch_docs, embedding_model, persist_directory=db_path)
        print(f"      Indexed {len(batch_docs)} blocks.")

print(f"\n Database Path: {db_path}")

<h4> <strong>  4) Semantic Retrieval for Financial Report Question Answering: </strong> </h4>

* defines a query function that enhances user questions with financial keywords and selects relevant sections for reasoning-based queries
* filters retrieval by ticker and year
* performs similarity search
* aggregates top-k document chunks
* returns a structured context for downstream analysis or LLM inference


In [ ]:
db_path = os.path.join(project_folder, "chroma_db_financial_semantic")
embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = Chroma(persist_directory=db_path, embedding_function=embedding_model)

def query_financial_report_inference(query: str, ticker: str, year: int):
    search_query = query
    if any(x in query.lower() for x in ["why", "explain", "change", "reason"]):
        search_query += " Management's Discussion and Analysis Liquidity and Capital Resources"
    
    enhanced_query = f"{query} financial statements balance sheet operations discussion analysis"
    
    print(f"   Modular Search: '{query}' for {ticker} ({year})")
    
    filter_dict = {"$and": [{"ticker": ticker}, {"year": year}]}
    
    try:
        results = vectorstore.similarity_search(enhanced_query, k=20, filter=filter_dict)
        
        if not results:
            return f"No records found for {ticker} in {year}."
        
        context = f"--- RESULTS FOR {ticker} ({year}) ---\n"
        for i, doc in enumerate(results):
            context += f"\n[Document Chunk {i+1}]\n{doc.page_content}\n"
            
        return context

    except Exception as e:
        return f"Retrieval Error: {e}"

<h4> <strong>  5) Multi-Tool LLM Agent for Financial Research and Calculation: </strong> </h4>

* sets up a **Python REPL-based calculator** for robust execution of financial math
* defines structured tools: one for **semantic retrieval of 10-K reports** and another for performing **financial calculations**
* constructs a **Lead Financial Research Agent** with a systematic procedure: analyze query, retrieve data, calculate ratios, explain formulas, and perform qualitative reasoning, using a Gemini LLM and sequential tool execution


In [ ]:
repl = PythonREPLTool()

def f_calculator(code: str):
    try:
        if "=" not in code and "print" not in code and len(code) < 50:
            return str(eval(code))
        return repl.run(code)
    except Exception as e:
        return f"Math Error: {e}"

class ReportSearchInput(BaseModel):
    query: str = Field(description="Financial topic to investigate.")
    ticker: str = Field(description="Stock ticker (e.g. AAPL).")
    year: int = Field(description="Fiscal year (e.g. 2023).")

financial_tool = StructuredTool.from_function(func=query_financial_report_inference, name="search_10k_reports", description="Searches financial reports for tables and management discussion. Always provide ticker and year.", args_schema=ReportSearchInput)
python_tool = Tool(name="python_calculator", func=f_calculator, description="Useful for performing financial calculations, ratios, and math. Input should be valid python code.")
tools = [financial_tool, python_tool]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0, max_output_tokens=4000)

system_prompt = """
    You are a Lead Financial Research Agent. You solve complex queries by creating a 'Research Plan'.

    STRICT OPERATING PROCEDURE:
    1. **Analyze the Goal:** Determine if the user wants a single data point, a calculation, or a multi-year comparison.
    2. **Draft a Plan:** - Step 1: Identify and retrieve the specific 10-K tables needed for all years involved.
    - Step 2: Use the python_calculator for math.
    - Step 3: Search for qualitative sections (MD&A, Risk Factors) to find the 'Why'.
    3. **Execute:** Run the steps sequentially. 
    4. **Self-Correct:** If a table looks incomplete, try a more specific search query.

    RATIO FORMULAS:
    - Acid Test: (Total Current Assets - Inventories) / Total Current Liabilities.
    - Current Ratio: Total Current Assets / Total Current Liabilities.
    - Net Margin: Net Income / Total Net Sales.

    Always explain the formula you used and list the raw numbers found before showing the final result.
    """

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"), 
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, max_iterations=10, return_intermediate_steps=True)

print(" Pipeline Complete")

<h4> <strong>  6) Interactive Financial Analyst Chat Loop: </strong> </h4>

* starts a **continuous REPL-style interface** where users can input financial queries until they type exit
* passes user queries to the **multi-tool LLM agent**, retrieves results, and processes raw outputs into a clean, readable answer

In [ ]:
print(" Financial Analyst is ready. Type 'exit' to quit.")

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['exit', 'quit', 'q']:
        break
    
    result = agent_executor.invoke({"input": user_input})
    raw_output = result["output"]
    
    if isinstance(raw_output, list):
        parts = []
        for block in raw_output:
            if isinstance(block, dict):
                content = block.get('text') or block.get('content')
                if content:
                    parts.append(content)
            elif isinstance(block, str):
                parts.append(block)
        clean_answer = "".join(parts)
    else:
        clean_answer = str(raw_output)

    clean_answer = clean_answer.replace("Reasoning:", "").replace("Thought:", "")
    
    print("\n" + "="*60)
    print("         Generated Answer ")
    print("="*60)
    print(clean_answer.strip())
    print("="*60)

<h4> <strong>  7) Financial Analyzer Evaluation & Benchmarking Suite: </strong> </h4>

* defines a comprehensive **evaluation dataset** covering three categories: pure calculation, pure textual reasoning, and mixed calculation + reasoning across multiple tickers and years
* specifies **ground-truth answers** and the set of tools expected for each query

In [ ]:
# --- ANALYSER EVALUATION & BENCHMARKING SUITE ---
eval_set = [
    # --- PURE CALCULATION ---
    {"ticker": "AAPL", "year": 2023, "question": "What is the current ratio for Apple in 2023?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "0.988 (Current Assets: $143,566M / Current Liabilities: $145,308M)"},
    {"ticker": "AMZN", "year": 2022, "question": "Calculate the Net Profit Margin for Amazon in 2022.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "-0.53% (Net Loss: $2,722M / Net Sales: $514,005M)"},
    {"ticker": "TSLA", "year": 2023, "question": "What was the year-over-year growth in Tesla's total assets between 2022 and 2023?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "29.49% (2023: $106,618M vs 2022: $82,338M)"},
    {"ticker": "MSFT", "year": 2023, "question": "Calculate the Debt-to-Equity ratio for Microsoft for 2023.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "0.229 (Total Debt: $47,193M / Total Equity: $206,223M)"},
    {"ticker": "NVDA", "year": 2024, "question": "What is the inventory turnover for Nvidia in 2024?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "3.18x (COGS: $16,621M / Avg Inventory: $5,221M)"},

    # --- PURE TEXTUAL REASONING ---
    {"ticker": "META", "year": 2023, "question": "What does management identify as the primary risk regarding competition in 2023?", 
     "gt_tools": ["search_10k_reports"], 
     "gt_answer": "Intense competition from short-form video platforms (TikTok), alternative social networks, and the impact of 'Family of Apps' engagement shifts."},
    {"ticker": "GOOGL", "year": 2022, "question": "Explain Google's strategy for AI investment as mentioned in the 2022 MD&A.", 
     "gt_tools": ["search_10k_reports"], 
     "gt_answer": "An 'AI-first' focus, integrating generative AI into Search/Cloud (Bard) and investing in DeepMind and high-performance compute infrastructure."},
    {"ticker": "NFLX", "year": 2023, "question": "What reasons does Netflix give for the change in its content spending strategy in 2023?", 
     "gt_tools": ["search_10k_reports"], 
     "gt_answer": "Focusing on profitability and quality over quantity, compounded by production timing delays caused by industry strikes."},
    {"ticker": "V", "year": 2023, "question": "How does Visa describe the impact of cross-border travel recovery on its revenue?", 
     "gt_tools": ["search_10k_reports"], 
     "gt_answer": "Cross-border volume increased 22%, driving a 29% growth in international transaction revenues as pandemic travel restrictions eased."},
    {"ticker": "JPM", "year": 2023, "question": "What are the key cybersecurity risks highlighted by JP Morgan in 2023?", 
     "gt_tools": ["search_10k_reports"], 
     "gt_answer": "Sophisticated AI-driven phishing, ransomware extortion, and vulnerabilities in third-party software supply chains."},

    # --- MIXED (CALCULATION + REASONING) ---
    {"ticker": "AAPL", "year": 2023, "question": "Did Apple's R&D as a % of sales increase in 2023, and what drove that change?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Yes, it increased to 7.8% (from 6.7% in 2022), driven primarily by headcount-related expenses and product development engineering costs."},
    {"ticker": "AMZN", "year": 2023, "question": "Calculate the ROA for 2023 and explain management's view on asset utilization efficiency.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "ROA was 6.12%. Management prioritized 'regionalization' of the fulfillment network to reduce transportation distances and improve unit efficiency."},
    {"ticker": "TSLA", "year": 2022, "question": "What was Tesla's gross margin in 2022, and what specific headwinds impacted it?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Gross margin was 25.6%. Headwinds included high lithium prices, raw material inflation, and the logistical challenges of ramping new factories in Berlin and Austin."},
    {"ticker": "NVDA", "year": 2024, "question": "Analyze the change in NVDA's cash position from 2023 to 2024 and why they increased it.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Cash/equivalents grew by $12.6B to $25.9B, driven by record-breaking Data Center revenue from H100 GPU demand."},
    {"ticker": "META", "year": 2023, "question": "How did the family of apps' operating margin change in 2023 and what was the role of 'Year of Efficiency'?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Margin increased from 34% to 47%. The 'Year of Efficiency' involved a 22% headcount reduction and consolidation of real estate facilities."},
    {"ticker": "MSFT", "year": 2023, "question": "What is the Cloud revenue growth rate for 2023, and how does management explain the Azure demand?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Growth was 22% (to $111B). Management attributed Azure demand to a 'structural shift' as customers migrated workloads to AI-optimized infrastructure."},
    {"ticker": "NFLX", "year": 2022, "question": "Calculate free cash flow for 2022 and explain why it differed from Net Income.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "FCF was $1.6B while Net Income was $4.5B. The difference is due to the non-cash amortization of content assets vs. the actual cash spent on production."},
    {"ticker": "GOOGL", "year": 2023, "question": "What was the effective tax rate in 2023, and what discrete tax items influenced it?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "14% (down from 16%). Influenced by a $1.2B discrete tax benefit related to a valuation allowance reversal and geographic income mix."},
    {"ticker": "V", "year": 2022, "question": "Calculate the dividend payout ratio and explain their capital return philosophy.", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "Payout ratio was ~23%. Philosophy is to prioritize organic growth and acquisitions first, followed by consistent dividend growth and opportunistic buybacks."},
    {"ticker": "JPM", "year": 2022, "question": "What was the CET1 capital ratio in 2022 and how did the First Republic acquisition impact their outlook?", 
     "gt_tools": ["search_10k_reports", "python_calculator"], 
     "gt_answer": "CET1 was 13.2%. Note: The 2022 report does not mention First Republic as the acquisition occurred in May 2023 (this tests the agent's year-filtering accuracy)."}
]

<h4> <strong>  8) Automated Evaluation: </strong> </h4>

* implements an **evaluation loop** over the benchmark queries: executes the agent, extracts actual tool usage, checks against expected tools, and records both system answers and tool-path correctness
* calculates **functional tool-call accuracy**


In [ ]:
def clean_output(agent_response):
    raw = agent_response.get("output", "")

    if isinstance(raw, list):
        parts = []
        for block in raw:
            if isinstance(block, dict):
                content = block.get('text') or block.get('content')
                if content:
                    parts.append(str(content))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts).strip()

    raw_str = str(raw)
    
    if "type': 'text'" in raw_str and "'text':" in raw_str:
        
        cleaned = re.sub(r"^\[\s*\{\s*'type':\s*'text',\s*'text':\s*[\"']", "", raw_str)
        cleaned = re.split(r"[\"'],\s*'extras':", cleaned)[0]        
        cleaned = cleaned.replace(r"\n", "\n").replace(r"\'", "'").replace(r'\"', '"')
        return cleaned.strip()

    return raw_str

def evaluate_analyzer(eval_list):
    results = []
    total_tool_correct = 0

    print(f"🕵️ Analyzing {len(eval_list)} queries...\n")

    for i, item in enumerate(eval_list):
        print(f"[{i+1}/{len(eval_list)}] Running {item['ticker']} {item['year']}...")
        
        try:
            response = agent_executor.invoke({"input": item['question']})            
            final_answer = clean_output(response)
            
            actual_tools = []
            if "intermediate_steps" in response:
                actual_tools = [step[0].tool for step in response["intermediate_steps"]]
            
            tool_accuracy = all(t in actual_tools for t in item['gt_tools'])
            if tool_accuracy: 
                total_tool_correct += 1

            results.append({
                "Company": f"{item['ticker']} ({item['year']})",
                "Ground_Truth_Answer": item['gt_answer'],
                "System_Answer": final_answer,
                "Tool_Path_Score": " Pass" if tool_accuracy else " Fail"
            })
            
        except Exception as e:
            print(f"    Error: {e}")
            results.append({
                "Company": f"{item['ticker']} ({item['year']})",
                "Ground_Truth_Answer": item['gt_answer'],
                "System_Answer": f"Error: {str(e)}",
                "Tool_Path_Score": " Fail"
            })

    avg_score = (total_tool_correct / len(eval_list)) * 100
    return pd.DataFrame(results), avg_score

eval_df, tool_score = evaluate_analyzer(eval_set)

print("\n" + "="*60)
print(f" PERFORMANCE REPORT")
print(f"Functional Tool Call Accuracy: {tool_score:.2f}%")
print("="*60)

pd.set_option('display.max_colwidth', None)
display(eval_df[['Company', 'Ground_Truth_Answer', 'System_Answer', 'Tool_Path_Score']])